In [28]:
# === Cell 1: Imports & Device Setup ===
import os, sys, json, hashlib, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter, defaultdict
from pathlib import Path
from tqdm import tqdm

# PyG imports
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.utils import k_hop_subgraph, scatter

# Device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

Device: mps


In [14]:
# === Cell 2: Load Stage 1 Graphs & Deduplicate ===
splits = torch.load('ANIDC/dataset/stage1_splits_7class/splits.pt', map_location='cpu', weights_only=False)

balanced_train = splits['balanced_train']
val_data = splits['val_data']
test_data = splits['test_data']

# Deduplicate balanced_train (oversampling created copies)
# Use (num_nodes, num_edges, first few feature values) as fingerprint
seen = set()
unique_train = []
for g in balanced_train:
    key = (g.x.shape[0], g.edge_index.shape[1], round(g.x.sum().item(), 4))
    if key not in seen:
        seen.add(key)
        unique_train.append(g)

# Combine all unique graphs
all_unique_graphs = unique_train + list(val_data) + list(test_data)

# Ensure y_attack exists on all graphs
for g in all_unique_graphs:
    if not hasattr(g, 'y_attack') or g.y_attack is None:
        if hasattr(g, 'y') and g.y is not None:
            g.y_attack = g.y if g.y.dim() == 0 else g.y[0:1]
        else:
            g.y_attack = torch.tensor([0], dtype=torch.long)

graph_labels = [g.y_attack.item() for g in all_unique_graphs]

id_to_attack = {0: "Benign", 1: "DoS", 2: "DDoS", 3: "PortScan", 4: "BruteForce", 5: "WebAttack", 6: "Bot/Other"}

print(f"Unique graphs: {len(all_unique_graphs)}")
print(f"  From balanced_train (deduped): {len(unique_train)}")
print(f"  From val: {len(val_data)}")
print(f"  From test: {len(test_data)}")
print(f"\nClass distribution:")
for cls, count in sorted(Counter(graph_labels).items()):
    print(f"  {id_to_attack.get(cls, f'Class {cls}')}: {count}")

Unique graphs: 2934
  From balanced_train (deduped): 2053
  From val: 440
  From test: 441

Class distribution:
  Benign: 2470
  DoS: 26
  DDoS: 20
  PortScan: 26
  BruteForce: 122
  WebAttack: 68
  Bot/Other: 202


## Section 2: Dataset Expansion (2.9K → 10K+)
Ego-graph extraction + augmentation to reach 10K graphs for Stage 2 training.

In [15]:
# === Cell 3: Ego-Graph Extraction ===
K_HOPS = 2
MAX_EGOS_PER_GRAPH = 5
MIN_EGO_NODES = 4
MIN_PARENT_NODES = 8

ego_graphs = []

for parent_idx, data in enumerate(tqdm(all_unique_graphs, desc="Extracting ego graphs")):
    if data.num_nodes < MIN_PARENT_NODES:
        continue
    if data.edge_index.size(1) == 0:
        continue

    # Find top-k highest degree nodes
    deg = torch.zeros(data.num_nodes, dtype=torch.long)
    deg.scatter_add_(0, data.edge_index[0], torch.ones(data.edge_index.size(1), dtype=torch.long))
    deg.scatter_add_(0, data.edge_index[1], torch.ones(data.edge_index.size(1), dtype=torch.long))
    top_nodes = deg.argsort(descending=True)[:MAX_EGOS_PER_GRAPH]

    for center in top_nodes.tolist():
        subset, sub_edge_index, mapping, edge_mask = k_hop_subgraph(
            node_idx=center,
            num_hops=K_HOPS,
            edge_index=data.edge_index,
            relabel_nodes=True,
            num_nodes=data.num_nodes,
        )

        n_sub = len(subset)
        if n_sub < MIN_EGO_NODES or n_sub >= data.num_nodes:
            continue

        ego_data = Data(
            x=data.x[subset],
            edge_index=sub_edge_index,
            y_attack=data.y_attack.clone(),
            y=data.y_attack.clone(),
        )
        if hasattr(data, 'node_attack_labels'):
            ego_data.node_attack_labels = data.node_attack_labels[subset]

        ego_graphs.append(ego_data)

print(f"\nOriginal graphs: {len(all_unique_graphs)}")
print(f"Ego graphs extracted: {len(ego_graphs)}")
print(f"Ego label distribution:")
ego_labels = [g.y_attack.item() for g in ego_graphs]
for cls, count in sorted(Counter(ego_labels).items()):
    print(f"  {id_to_attack.get(cls, f'Class {cls}')}: {count}")

Extracting ego graphs: 100%|████████████| 2934/2934 [00:00<00:00, 3336.70it/s]


Original graphs: 2934
Ego graphs extracted: 13019
Ego label distribution:
  Benign: 10859
  DoS: 125
  DDoS: 96
  PortScan: 126
  BruteForce: 587
  WebAttack: 311
  Bot/Other: 915


In [16]:
# === Cell 4: Graph Augmentation (balance underrepresented classes) ===
def augment_graph(graph, edge_drop_rate=0.15, noise_std=0.05):
    """Create augmented copy: drop edges + add feature noise."""
    new_x = graph.x.clone() + torch.randn_like(graph.x) * noise_std

    # Edge dropout
    num_edges = graph.edge_index.size(1)
    keep_mask = torch.rand(num_edges) > edge_drop_rate
    if keep_mask.sum() < 2:  # keep at least 2 edges
        keep_mask[:2] = True
    new_edge_index = graph.edge_index[:, keep_mask]

    aug = Data(
        x=new_x,
        edge_index=new_edge_index,
        y_attack=graph.y_attack.clone(),
        y=graph.y_attack.clone(),
    )
    return aug

# Combine original + ego first
combined = all_unique_graphs + ego_graphs
combined_labels = [g.y_attack.item() for g in combined]
print(f"Before augmentation: {len(combined)} graphs")

# Check if we need augmentation to reach 10K
TARGET = 10000
if len(combined) < TARGET:
    deficit = TARGET - len(combined)
    print(f"Need {deficit} more graphs via augmentation")

    # Augment underrepresented classes more
    label_counts = Counter(combined_labels)
    max_count = max(label_counts.values())

    augmented = []
    np.random.seed(42)

    # First pass: balance classes
    for cls in sorted(label_counts.keys()):
        cls_graphs = [g for g in combined if g.y_attack.item() == cls]
        needed = max(0, max_count - len(cls_graphs))
        if needed > 0:
            for _ in range(needed):
                src = cls_graphs[np.random.randint(len(cls_graphs))]
                augmented.append(augment_graph(src))

    # Second pass: if still short, augment randomly
    while len(combined) + len(augmented) < TARGET:
        src = combined[np.random.randint(len(combined))]
        augmented.append(augment_graph(src))

    print(f"Augmented graphs created: {len(augmented)}")
    combined = combined + augmented
else:
    print("Already at 10K+, no augmentation needed")

print(f"\nTotal after augmentation: {len(combined)}")
combined_labels = [g.y_attack.item() for g in combined]
print("Final distribution:")
for cls, count in sorted(Counter(combined_labels).items()):
    print(f"  {id_to_attack.get(cls, f'Class {cls}')}: {count}")

Before augmentation: 15953 graphs
Already at 10K+, no augmentation needed

Total after augmentation: 15953
Final distribution:
  Benign: 13329
  DoS: 151
  DDoS: 116
  PortScan: 152
  BruteForce: 709
  WebAttack: 379
  Bot/Other: 1117


In [17]:
# === Cell 5: Save Combined Graphs ===
save_path = 'ANIDC/dataset/stage2_all_graphs_77dim.pt'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(combined, save_path)
print(f"Saved {len(combined)} graphs to {save_path}")
print(f"Node feature dim: {combined[0].x.shape[1]}")
print(f"Sample graph: {combined[0].x.shape[0]} nodes, {combined[0].edge_index.shape[1]} edges, label={combined[0].y_attack.item()}")

Saved 15953 graphs to ANIDC/dataset/stage2_all_graphs_77dim.pt
Node feature dim: 77
Sample graph: 148 nodes, 200 edges, label=2


## Section 3: Template-Based Text Generation & JSONL Export
Generate text descriptions for each graph using attack-specific templates, then save as JSONL for Flan-T5 augmentation.

In [18]:
# === Cell 6: Generate Text Descriptions ===
import networkx as nx

def extract_graph_statistics(graph):
    """Extract stats from a PyG graph for template filling."""
    num_nodes = graph.x.size(0)
    num_edges = graph.edge_index.size(1)

    edge_index = graph.edge_index.cpu().numpy()
    G = nx.DiGraph()
    G.add_nodes_from(range(num_nodes))
    if num_edges > 0:
        G.add_edges_from(edge_index.T)

    degrees = [deg for _, deg in G.degree()]
    stats = {
        'num_nodes': num_nodes,
        'num_edges': num_edges,
        'avg_degree': num_edges / num_nodes if num_nodes > 0 else 0,
        'density': nx.density(G) if num_nodes > 1 else 0,
        'max_degree': max(degrees) if degrees else 0,
        'num_components': nx.number_weakly_connected_components(G) if num_nodes > 0 else 0,
    }

    # Attack label
    if hasattr(graph, 'y_attack') and graph.y_attack is not None:
        stats['attack_label'] = graph.y_attack.item()
    elif hasattr(graph, 'y') and graph.y is not None:
        stats['attack_label'] = graph.y.item() if graph.y.dim() == 0 else graph.y[0].item()
    else:
        stats['attack_label'] = -1

    return stats

def get_attack_type_name(label):
    mapping = {0: "Benign", 1: "DoS", 2: "DDoS", 3: "PortScan",
               4: "BruteForce", 5: "WebAttack", 6: "Botnet", 7: "Infiltration", -1: "Unknown"}
    return mapping.get(label, "Unknown")

def generate_template_description(stats):
    """Generate text from graph stats using attack-specific templates."""
    attack_type = get_attack_type_name(stats['attack_label'])
    n, e = stats['num_nodes'], stats['num_edges']
    ad, d = stats['avg_degree'], stats['density']
    nc, md = stats['num_components'], stats['max_degree']

    templates = {
        "Benign": [
            f"Normal network traffic with {n} nodes and {e} connections. The network shows typical communication patterns with average degree {ad:.1f}.",
            f"Benign traffic pattern involving {n} network entities. Graph density is {d:.3f}, indicating normal connectivity.",
            f"Regular network activity with {n} hosts exchanging {e} packets. Network topology shows {nc} component(s) with standard structure.",
        ],
        "DoS": [
            f"Denial of Service attack detected with {n} nodes and {e} connections. Unusually high degree centrality (avg {ad:.1f}) indicates flooding behavior.",
            f"DoS attack pattern: {n} network nodes with concentrated traffic. Maximum degree of {md} suggests targeted flooding.",
            f"Network under DoS attack with {e} malicious connections. Topology shows {nc} component(s) with attack concentration.",
        ],
        "DDoS": [
            f"Distributed Denial of Service attack involving {n} nodes. High edge count ({e}) and density ({d:.3f}) indicate coordinated botnet activity.",
            f"DDoS attack with distributed sources: {n} participating nodes. Network shows {nc} component(s), suggesting multiple attack vectors.",
            f"Coordinated DDoS pattern with {e} connections from distributed sources. Average degree {ad:.1f} reflects botnet communication structure.",
        ],
        "PortScan": [
            f"Port scanning activity detected across {n} network nodes. High connectivity ({e} edges) from scanning probes.",
            f"Network reconnaissance via port scan: {n} targets with {e} probe attempts. Topology reveals {nc} scanning pattern(s).",
            f"Port scan attack with {e} connection attempts. Graph structure shows systematic probing across {n} hosts.",
        ],
        "BruteForce": [
            f"Brute force attack pattern with {n} nodes involved. Repeated connection attempts ({e} edges) to authentication services.",
            f"Authentication brute force attack: {e} login attempts. Network topology shows {nc} attack vector(s) targeting {n} nodes.",
            f"Brute force credential attack with {n} network entities. High edge count ({e}) indicates repeated authentication probes.",
        ],
        "WebAttack": [
            f"Web application attack detected involving {n} nodes. HTTP/HTTPS traffic shows {e} malicious request patterns.",
            f"Web-based attack: {e} HTTP connections to {n} targets. Graph density {d:.3f} suggests targeted web exploitation.",
            f"Web application attack pattern with {n} hosts and {e} requests. Topology reveals {nc} exploitation attempt(s).",
        ],
        "Botnet": [
            f"Botnet C&C communication detected: {n} compromised hosts. Command-and-control traffic shows {e} coordination edges.",
            f"Botnet activity with {n} infected nodes and {e} C&C connections. Network structure shows {nc} botnet cluster(s).",
            f"Bot network pattern: {e} communication channels among {n} bots. Average degree {ad:.1f} reflects botnet topology.",
        ],
        "Infiltration": [
            f"Network infiltration detected: {n} nodes with {e} suspicious connections. Stealthy communication pattern with low density ({d:.3f}).",
            f"Infiltration attack involving {n} network entities. Graph shows {nc} component(s) with covert data exfiltration.",
            f"Lateral movement and infiltration: {e} connections across {n} hosts. Topology suggests advanced persistent threat behavior.",
        ],
    }

    t = templates.get(attack_type, [
        f"Network traffic graph with {n} nodes and {e} edges. Average degree {ad:.1f}, density {d:.3f}.",
        f"Network topology: {n} hosts, {e} connections. Structure contains {nc} connected component(s).",
    ])

    # Deterministic selection via hash
    hash_val = int(hashlib.md5(str(stats).encode()).hexdigest(), 16)
    return t[hash_val % len(t)]

# Generate graph-text pairs
samples = []
for i, graph in enumerate(tqdm(combined, desc="Generating text descriptions")):
    stats = extract_graph_statistics(graph)
    text = generate_template_description(stats)
    attack_name = get_attack_type_name(stats['attack_label'])

    sample = {
        'graph_id': f"graph_{i:06d}",
        'node_features': graph.x.cpu().tolist(),
        'edge_index': graph.edge_index.cpu().tolist(),
        'text': text,
        'attack_type': attack_name,
        'metadata': {
            'num_nodes': stats['num_nodes'],
            'num_edges': stats['num_edges'],
            'avg_degree': round(stats['avg_degree'], 3),
            'density': round(stats['density'], 4),
            'num_components': stats['num_components'],
            'source': 'template_generation',
            'augmented': False,
        }
    }
    samples.append(sample)

print(f"\nGenerated {len(samples)} graph-text pairs")
print(f"\nSample entry:")
print(f"  ID: {samples[0]['graph_id']}")
print(f"  Attack: {samples[0]['attack_type']}")
print(f"  Text: {samples[0]['text'][:100]}...")
print(f"  Nodes: {len(samples[0]['node_features'])}, Edges: {len(samples[0]['edge_index'][0])}")

Generating text descriptions: 100%|███| 15953/15953 [00:03<00:00, 4570.91it/s]


Generated 15953 graph-text pairs

Sample entry:
  ID: graph_000000
  Attack: DDoS
  Text: DDoS attack with distributed sources: 148 participating nodes. Network shows 3 component(s), suggest...
  Nodes: 148, Edges: 200


In [19]:
# === Cell 7: Save JSONL + Train/Val/Test Split ===
from sklearn.model_selection import train_test_split

output_dir = Path('ANIDC/dataset/stage2_dataset_77dim_v2')
output_dir.mkdir(parents=True, exist_ok=True)

# Stratified split 70/15/15
attack_types = [s['attack_type'] for s in samples]
train_samples, temp_samples = train_test_split(
    samples, train_size=0.7, stratify=attack_types, random_state=42
)
temp_attacks = [s['attack_type'] for s in temp_samples]
val_samples, test_samples = train_test_split(
    temp_samples, train_size=0.5, stratify=temp_attacks, random_state=42
)

def save_jsonl(data, path):
    with open(path, 'w') as f:
        for s in data:
            f.write(json.dumps(s) + '\n')
    print(f"  Saved {len(data)} samples to {path}")

save_jsonl(train_samples, output_dir / 'train.jsonl')
save_jsonl(val_samples, output_dir / 'val.jsonl')
save_jsonl(test_samples, output_dir / 'test.jsonl')

# Also save full set for Flan-T5 augmentation
save_jsonl(samples, output_dir / 'all_templates.jsonl')

print(f"\nSplit distribution:")
for name, split in [("Train", train_samples), ("Val", val_samples), ("Test", test_samples)]:
    dist = Counter(s['attack_type'] for s in split)
    print(f"  {name}: {len(split)} samples - {dict(sorted(dist.items()))}")

print(f"\nTo augment with Flan-T5, run on Kaggle:")
print(f"  python 02_augment_with_flan_kaggle.py --input train.jsonl --output train_augmented.jsonl --model google/flan-t5-base")

  Saved 11167 samples to ANIDC/dataset/stage2_dataset_77dim_v2/train.jsonl
  Saved 2393 samples to ANIDC/dataset/stage2_dataset_77dim_v2/val.jsonl
  Saved 2393 samples to ANIDC/dataset/stage2_dataset_77dim_v2/test.jsonl
  Saved 15953 samples to ANIDC/dataset/stage2_dataset_77dim_v2/all_templates.jsonl

Split distribution:
  Train: 11167 samples - {'Benign': 9330, 'Botnet': 782, 'BruteForce': 496, 'DDoS': 81, 'DoS': 106, 'PortScan': 107, 'WebAttack': 265}
  Val: 2393 samples - {'Benign': 2000, 'Botnet': 168, 'BruteForce': 106, 'DDoS': 17, 'DoS': 22, 'PortScan': 23, 'WebAttack': 57}
  Test: 2393 samples - {'Benign': 1999, 'Botnet': 167, 'BruteForce': 107, 'DDoS': 18, 'DoS': 23, 'PortScan': 22, 'WebAttack': 57}

To augment with Flan-T5, run on Kaggle:
  python 02_augment_with_flan_kaggle.py --input train.jsonl --output train_augmented.jsonl --model google/flan-t5-base


## Section 4: Dataset & DataLoaders
Load JSONL graph-text pairs into PyG format for training.

In [20]:
# === Cell 8: GraphTextDataset & Collate Function ===
from torch.utils.data import Dataset as TorchDataset

class GraphTextDataset(TorchDataset):
    """Dataset of (graph, text) pairs from JSONL."""

    ATTACK_TYPE_TO_ID = {
        'Benign': 0, 'Bot/Other': 1, 'Botnet': 1, 'BruteForce': 2,
        'DDoS': 3, 'DoS': 4, 'Infiltration': 5, 'PortScan': 6, 'WebAttack': 7,
        'Unknown': 8,
    }

    def __init__(self, data_path, max_samples=None, load_auxiliary_labels=True):
        self.data_path = Path(data_path)
        self.load_auxiliary_labels = load_auxiliary_labels

        # Load JSONL
        self.samples = []
        with open(self.data_path, 'r') as f:
            for line in f:
                if max_samples and len(self.samples) >= max_samples:
                    break
                self.samples.append(json.loads(line.strip()))
        print(f"Loaded {len(self.samples)} samples from {self.data_path.name}")

        # Build attack mapping from data
        attack_types = sorted(set(s.get('attack_type', 'Unknown') for s in self.samples))
        self.attack_type_to_id = {at: i for i, at in enumerate(attack_types)}

        # Cache
        self.graph_cache = [None] * len(self.samples)

    def _parse_graph(self, sample):
        node_features = torch.tensor(sample['node_features'], dtype=torch.float)
        edge_index = torch.tensor(sample['edge_index'], dtype=torch.long)

        graph = Data(x=node_features, edge_index=edge_index)
        graph.graph_id = sample.get('graph_id', 'unknown')
        graph.attack_type = sample.get('attack_type', 'Unknown')

        if self.load_auxiliary_labels:
            attack_type = sample.get('attack_type', 'Unknown')
            graph.y_attack = torch.tensor(self.attack_type_to_id.get(attack_type, 0), dtype=torch.long)

            num_nodes = node_features.shape[0]
            num_edges = edge_index.shape[1]
            max_edges = num_nodes * (num_nodes - 1) / 2 if num_nodes > 1 else 1
            density = num_edges / max_edges if max_edges > 0 else 0.0

            graph.num_nodes_label = torch.tensor([num_nodes], dtype=torch.float32)
            graph.num_edges_label = torch.tensor([num_edges], dtype=torch.float32)
            graph.density_label = torch.tensor([density], dtype=torch.float32)

        return graph

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.graph_cache[idx] is not None:
            graph = self.graph_cache[idx]
        else:
            graph = self._parse_graph(self.samples[idx])
            self.graph_cache[idx] = graph
        return graph, self.samples[idx]['text']

    def get_attack_distribution(self):
        return dict(Counter(s.get('attack_type', 'Unknown') for s in self.samples))

    def get_num_attack_classes(self):
        return len(self.attack_type_to_id)


def collate_graph_text(batch):
    """Collate (graph, text) pairs into (Batch, List[str])."""
    graphs, texts = zip(*batch)
    batch_graphs = Batch.from_data_list(list(graphs))
    return batch_graphs, list(texts)

print("GraphTextDataset and collate_graph_text defined.")

GraphTextDataset and collate_graph_text defined.


In [21]:
# === Cell 8.5: Data Quality Filter ===
# 48% of Flan-T5 augmented texts have hallucinated numbers (wrong node/edge counts).
# Strategy: fall back to original template text for bad samples (preserve all samples).

import re

def filter_augmented_data(jsonl_path, output_path=None):
    """Filter low-quality augmented texts, falling back to originals."""
    if output_path is None:
        output_path = str(jsonl_path).replace('.jsonl', '_filtered.jsonl')

    kept_as_is = 0
    fell_back = 0
    removed = 0
    total = 0

    with open(jsonl_path, 'r') as fin, open(output_path, 'w') as fout:
        for line in fin:
            total += 1
            sample = json.loads(line.strip())
            text = sample['text']
            meta = sample.get('metadata', {})

            needs_fallback = False

            # Check 1: Garbled text (spurious quote wrapping from Flan-T5)
            if (text.startswith('"') and text.endswith('"')) or '[Deleted' in text:
                needs_fallback = True

            # Check 2: Too short
            if len(text) < 20:
                needs_fallback = True

            # Check 3: Key numbers missing from text
            num_nodes = meta.get('num_nodes')
            num_edges = meta.get('num_edges')
            if num_nodes is not None and num_edges is not None and meta.get('augmented', False):
                if not (str(num_nodes) in text and str(num_edges) in text):
                    needs_fallback = True

            if needs_fallback:
                original = meta.get('original_text')
                if original and len(original) >= 20:
                    sample['text'] = original
                    sample['metadata']['augmented'] = False
                    sample['metadata']['fallback_to_original'] = True
                    fell_back += 1
                else:
                    removed += 1
                    continue
            else:
                kept_as_is += 1

            fout.write(json.dumps(sample) + '\n')

    print(f"  {Path(jsonl_path).name}:")
    print(f"    Total: {total} | Kept: {kept_as_is} ({100*kept_as_is/total:.1f}%) | "
          f"Fell back: {fell_back} ({100*fell_back/total:.1f}%) | Removed: {removed}")
    return Path(output_path)

# Filter all splits
print("Data Quality Filter:")
data_dir_filter = Path('ANIDC/dataset/stage2_dataset_77dim_v2')
for split in ['train_augmented', 'val_augmented', 'test_augmented']:
    p = data_dir_filter / f'{split}.jsonl'
    if p.exists():
        filter_augmented_data(p)
    else:
        print(f"  Skipping {split} (not found)")

Data Quality Filter:
  train_augmented.jsonl:
    Total: 11167 | Kept: 5105 (45.7%) | Fell back: 6062 (54.3%) | Removed: 0
  val_augmented.jsonl:
    Total: 2393 | Kept: 1107 (46.3%) | Fell back: 1286 (53.7%) | Removed: 0
  test_augmented.jsonl:
    Total: 2393 | Kept: 1069 (44.7%) | Fell back: 1324 (55.3%) | Removed: 0


In [22]:
# === Cell 9: Create DataLoaders (class-balanced sampling) ===
from torch.utils.data import WeightedRandomSampler

data_dir = Path('ANIDC/dataset/stage2_dataset_77dim_v2')

# Prefer filtered > augmented > template-only
for suffix in ['_augmented_filtered', '_augmented', '']:
    train_path = data_dir / f'train{suffix}.jsonl'
    if train_path.exists():
        print(f"Using train: {train_path.name}")
        break

for suffix in ['_augmented_filtered', '_augmented', '']:
    val_path = data_dir / f'val{suffix}.jsonl'
    if val_path.exists():
        print(f"Using val:   {val_path.name}")
        break

for suffix in ['_augmented_filtered', '_augmented', '']:
    test_path = data_dir / f'test{suffix}.jsonl'
    if test_path.exists():
        print(f"Using test:  {test_path.name}")
        break

train_dataset = GraphTextDataset(train_path, load_auxiliary_labels=True)
val_dataset = GraphTextDataset(val_path, load_auxiliary_labels=True)
test_dataset = GraphTextDataset(test_path, load_auxiliary_labels=True)

num_attack_classes = train_dataset.get_num_attack_classes()
print(f"\nAttack classes: {num_attack_classes}")
print(f"Attack mapping: {train_dataset.attack_type_to_id}")

BATCH_SIZE = 32

# === Class-Balanced Sampling ===
# Weight each sample inversely to its class frequency.
# Current: ~27 Benign per batch. Target: ~4-5 per class per batch.
train_attack_types = [s.get('attack_type', 'Unknown') for s in train_dataset.samples]
train_class_counts = Counter(train_attack_types)
n_classes = len(train_class_counts)

sample_weights = []
for at in train_attack_types:
    w = 1.0 / (n_classes * train_class_counts[at])
    sample_weights.append(w)
sample_weights_t = torch.tensor(sample_weights, dtype=torch.float64)

train_sampler = WeightedRandomSampler(
    weights=sample_weights_t,
    num_samples=len(train_dataset),
    replacement=True
)

# sampler and shuffle are mutually exclusive
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
    collate_fn=collate_graph_text, num_workers=0
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_graph_text, num_workers=0
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_graph_text, num_workers=0
)

print(f"\nTrain: {len(train_dataset)} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_dataset)} samples ({len(val_loader)} batches)")
print(f"Test:  {len(test_dataset)} samples ({len(test_loader)} batches)")
print(f"Expected per class per batch: ~{BATCH_SIZE/n_classes:.1f}")

# Verify balanced batch
batch_graphs, batch_texts = next(iter(train_loader))
print(f"\nSample balanced batch:")
print(f"  Graphs: {batch_graphs.num_graphs} graphs, {batch_graphs.num_nodes} total nodes")
if hasattr(batch_graphs, 'y_attack'):
    batch_dist = Counter(batch_graphs.y_attack.tolist())
    id_to_name_local = {v: k for k, v in train_dataset.attack_type_to_id.items()}
    print(f"  Class dist: { {id_to_name_local.get(k, k): v for k, v in sorted(batch_dist.items())} }")
print(f"  Text[0]: {batch_texts[0][:80]}...")

Using train: train_augmented_filtered.jsonl
Using val:   val_augmented_filtered.jsonl
Using test:  test_augmented_filtered.jsonl
Loaded 11167 samples from train_augmented_filtered.jsonl
Loaded 2393 samples from val_augmented_filtered.jsonl
Loaded 2393 samples from test_augmented_filtered.jsonl

Attack classes: 7
Attack mapping: {'Benign': 0, 'Botnet': 1, 'BruteForce': 2, 'DDoS': 3, 'DoS': 4, 'PortScan': 5, 'WebAttack': 6}

Train: 11167 samples (349 batches)
Val:   2393 samples (75 batches)
Test:  2393 samples (75 batches)
Expected per class per batch: ~4.6

Sample balanced batch:
  Graphs: 32 graphs, 1922 total nodes
  Class dist: {'Benign': 8, 'Botnet': 4, 'BruteForce': 5, 'DDoS': 3, 'DoS': 3, 'PortScan': 2, 'WebAttack': 7}
  Text[0]: Benign traffic pattern involving 52 network entities. Graph density is 0.041, in...


## Section 5: Model Architecture
GATEncoder (frozen) + BERT (frozen) + CLIP-style projection heads + InfoNCE loss.

In [23]:
# === Cell 10: Stage 1 GNN Encoder (frozen) ===
class GATEncoderWrapper(nn.Module):
    """GATv2Conv encoder with .encode() and .latent_dim for Stage 2 compatibility."""

    def __init__(self, input_dim=77, hidden_dim=128, num_heads=4, dropout=0.2):
        super().__init__()
        self.latent_dim = hidden_dim
        self.conv1 = GATv2Conv(input_dim, hidden_dim, heads=num_heads, dropout=dropout)
        self.conv2 = GATv2Conv(hidden_dim * num_heads, hidden_dim, heads=num_heads, dropout=dropout)
        self.conv3 = GATv2Conv(hidden_dim * num_heads, hidden_dim, heads=1, dropout=dropout)

    def encode(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        x = F.elu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)
        return x  # [N, hidden_dim]

    def forward(self, x, edge_index):
        return self.encode(x, edge_index)

    @classmethod
    def from_checkpoint(cls, checkpoint, device='cpu'):
        config = checkpoint['model_config']
        model = cls(
            input_dim=config['input_dim'],
            hidden_dim=config['hidden_dim'],
            num_heads=config.get('num_heads', 4),
            dropout=config.get('dropout', 0.2),
        )
        model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        return model.to(torch.device(device))

# Load Stage 1 checkpoint
checkpoint = torch.load('ANIDC/checkpoints/stage1_77dim/best.pt', map_location='cpu', weights_only=False)
gnn_model = GATEncoderWrapper.from_checkpoint(checkpoint, device=str(device))

# Freeze GNN
for param in gnn_model.parameters():
    param.requires_grad = False

print(f"GNN loaded from epoch {checkpoint.get('epoch', '?')}")
print(f"  Config: {checkpoint['model_config']}")
print(f"  Latent dim: {gnn_model.latent_dim}")
print(f"  Parameters: {sum(p.numel() for p in gnn_model.parameters()):,} (all frozen)")

GNN loaded from epoch 20
  Config: {'model_type': 'GATEncoder', 'input_dim': 77, 'hidden_dim': 128, 'num_layers': 3, 'num_heads': 4, 'dropout': 0.2}
  Latent dim: 128
  Parameters: 738,816 (all frozen)


In [24]:
# === Cell 11: BERT Text Encoder (frozen) ===
from transformers import AutoModel, AutoTokenizer

class BERTEncoder:
    """BERT text encoder - extracts CLS token embeddings."""

    def __init__(self, model_name="bert-base-uncased", device=None):
        self._device = device or torch.device('cpu')
        print(f"Loading {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self._device)
        self.model.eval()
        self._hidden_dim = self.model.config.hidden_size
        self.freeze()
        print(f"  Hidden dim: {self._hidden_dim}, Parameters: {sum(p.numel() for p in self.model.parameters()):,}")

    def encode(self, texts):
        inputs = self.tokenizer(
            texts, padding=True, truncation=True, max_length=128, return_tensors='pt'
        ).to(self._device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        return outputs.last_hidden_state[:, 0, :]  # CLS token [B, 768]

    @property
    def hidden_dim(self):
        return self._hidden_dim

    @property
    def device(self):
        return self._device

    def freeze(self):
        for param in self.model.parameters():
            param.requires_grad = False

    def unfreeze(self):
        for param in self.model.parameters():
            param.requires_grad = True

text_encoder = BERTEncoder(device=device)

Loading bert-base-uncased...
  Hidden dim: 768, Parameters: 109,482,240


In [25]:
# === Cell 12: CrossAttentionBridgeV2.2 (QFormer + SigLIP + Soft Targets) ===

class QFormerBridge(nn.Module):
    """Lightweight Q-Former for graph-to-fixed-dim bottleneck (BLIP-2/MolCA style).

    Learnable query tokens cross-attend to variable-length node embeddings,
    producing a fixed-size graph representation that selectively captures
    alignment-relevant structural features.
    """

    def __init__(self, num_queries=4, hidden_dim=256, gnn_dim=128,
                 num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.num_queries = num_queries
        self.hidden_dim = hidden_dim

        # Learnable query tokens
        self.query_tokens = nn.Parameter(torch.randn(1, num_queries, hidden_dim) * 0.02)

        # Project GNN node embeddings to Q-Former hidden_dim
        self.input_proj = nn.Linear(gnn_dim, hidden_dim)

        # Cross-attention + FFN layers
        self.cross_attn_layers = nn.ModuleList()
        self.cross_attn_norms = nn.ModuleList()
        self.ffn_layers = nn.ModuleList()
        self.ffn_norms = nn.ModuleList()

        for _ in range(num_layers):
            self.cross_attn_layers.append(
                nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
            )
            self.cross_attn_norms.append(nn.LayerNorm(hidden_dim))
            self.ffn_layers.append(nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(hidden_dim * 4, hidden_dim), nn.Dropout(dropout),
            ))
            self.ffn_norms.append(nn.LayerNorm(hidden_dim))

    def forward(self, node_embs, batch_idx):
        """
        Args:
            node_embs: [N_total, gnn_dim] all node embeddings (flat)
            batch_idx: [N_total] integer mapping each node to its graph
        Returns:
            graph_embs: [B, hidden_dim]
        """
        B = batch_idx.max().item() + 1
        node_proj = self.input_proj(node_embs)  # [N_total, hidden_dim]

        # Pad into [B, max_nodes, hidden_dim]
        counts = torch.bincount(batch_idx, minlength=B)
        max_nodes = counts.max().item()

        padded = torch.zeros(B, max_nodes, self.hidden_dim, device=node_embs.device)
        key_padding_mask = torch.ones(B, max_nodes, dtype=torch.bool, device=node_embs.device)

        for i in range(B):
            mask_i = (batch_idx == i)
            n_i = mask_i.sum().item()
            padded[i, :n_i] = node_proj[mask_i]
            key_padding_mask[i, :n_i] = False

        # Expand queries
        queries = self.query_tokens.expand(B, -1, -1)

        # Cross-attention layers with residual
        for ca, ca_norm, ffn, ffn_norm in zip(
            self.cross_attn_layers, self.cross_attn_norms,
            self.ffn_layers, self.ffn_norms
        ):
            attn_out, _ = ca(query=queries, key=padded, value=padded,
                             key_padding_mask=key_padding_mask)
            queries = ca_norm(queries + attn_out)
            queries = ffn_norm(queries + ffn(queries))

        return queries.mean(dim=1)  # [B, hidden_dim]


class AuxiliaryHeads(nn.Module):
    """Auxiliary task heads: attack classification + property prediction + triplet."""

    def __init__(self, hidden_dim=256, num_attack_classes=7):
        super().__init__()
        self.attack_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden_dim // 2, num_attack_classes)
        )
        self.node_predictor = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1))
        self.edge_predictor = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1))
        self.density_predictor = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
        self.task_weights = {'attack': 0.4, 'property': 0.3, 'triplet': 0.3}

    def compute_loss(self, graph_embs, text_embs, batch):
        losses = {}
        metrics = {}

        if hasattr(batch, 'y_attack'):
            logits = self.attack_head(graph_embs)
            loss_atk = F.cross_entropy(logits, batch.y_attack, label_smoothing=0.1)
            losses['attack'] = loss_atk
            with torch.no_grad():
                acc = (logits.argmax(1) == batch.y_attack).float().mean().item()
            metrics['attack_cls_acc'] = acc

        if hasattr(batch, 'num_nodes_label'):
            pred_nodes = self.node_predictor(text_embs).squeeze(-1) / 100.0
            pred_edges = self.edge_predictor(text_embs).squeeze(-1) / 100.0
            pred_density = self.density_predictor(text_embs).squeeze(-1)
            loss_prop = (
                0.3 * F.mse_loss(pred_nodes, batch.num_nodes_label / 100.0) +
                0.3 * F.mse_loss(pred_edges, batch.num_edges_label / 100.0) +
                0.4 * F.mse_loss(pred_density, batch.density_label)
            )
            losses['property'] = loss_prop

        if hasattr(batch, 'y_attack') and graph_embs.size(0) > 2:
            pos_sim = F.cosine_similarity(graph_embs, text_embs, dim=1)
            neg_text = text_embs.roll(1, dims=0)
            neg_sim = F.cosine_similarity(graph_embs, neg_text, dim=1)
            loss_triplet = torch.clamp(0.5 + neg_sim - pos_sim, min=0.0).mean()
            losses['triplet'] = loss_triplet
            with torch.no_grad():
                metrics['triplet_violations'] = (neg_sim > pos_sim).float().mean().item()

        total = sum(self.task_weights.get(k, 0.3) * v for k, v in losses.items())
        return total, metrics


class CrossAttentionBridgeV2(nn.Module):
    """v2.2: QFormer + SigLIP + ConGraT soft targets."""

    def __init__(self, gnn_model, text_encoder, hidden_dim=256, dropout=0.1,
                 pooling='mean', use_auxiliary_tasks=True, num_attack_classes=7,
                 contrastive_weight=0.5, auxiliary_weight=0.5,
                 use_qformer=True, num_queries=4, num_qformer_layers=2,
                 soft_target_alpha=0.1):
        super().__init__()
        self.gnn = gnn_model
        self.text_encoder = text_encoder
        self.hidden_dim = hidden_dim
        self.pooling = pooling
        self.use_auxiliary_tasks = use_auxiliary_tasks
        self.contrastive_weight = contrastive_weight
        self.auxiliary_weight = auxiliary_weight
        self.soft_target_alpha = soft_target_alpha

        self.gnn_dim = gnn_model.latent_dim
        self.text_dim = text_encoder.hidden_dim

        # Q-Former bridge (replaces mean pooling)
        self.use_qformer = use_qformer
        if use_qformer:
            self.qformer = QFormerBridge(
                num_queries=num_queries, hidden_dim=hidden_dim,
                gnn_dim=self.gnn_dim, num_heads=4,
                num_layers=num_qformer_layers, dropout=dropout,
            )
            # QFormer outputs hidden_dim, so graph_proj: hidden_dim -> hidden_dim
            self.graph_proj = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
                nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            )
        else:
            # Original: mean pool outputs gnn_dim
            self.graph_proj = nn.Sequential(
                nn.Linear(self.gnn_dim, hidden_dim), nn.GELU(),
                nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            )

        self.text_proj = nn.Sequential(
            nn.Linear(self.text_dim, hidden_dim), nn.GELU(),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
        )

        # Learnable temperature
        self.temperature = nn.Parameter(torch.tensor(0.07))

        # Auxiliary tasks
        if use_auxiliary_tasks:
            self.auxiliary = AuxiliaryHeads(hidden_dim, num_attack_classes)

    def pool_graph(self, node_embs, batch_idx):
        """Fallback mean pooling (used when use_qformer=False)."""
        return scatter(node_embs, batch_idx, dim=0, reduce='mean')

    def forward(self, batch, texts, compute_auxiliary=True):
        # Encode nodes
        node_embs = self.gnn.encode(batch.x, batch.edge_index)

        # Graph embedding: QFormer or mean pool
        if self.use_qformer:
            graph_embs = self.qformer(node_embs, batch.batch)
        else:
            graph_embs = self.pool_graph(node_embs, batch.batch)
        graph_embs = self.graph_proj(graph_embs)

        # Text embedding
        text_embs = self.text_encoder.encode(texts)
        text_embs = self.text_proj(text_embs)

        # L2 normalize
        graph_normed = F.normalize(graph_embs, dim=-1)
        text_normed = F.normalize(text_embs, dim=-1)

        # SigLIP contrastive loss with optional soft targets
        attack_labels = batch.y_attack if hasattr(batch, 'y_attack') else None
        contrastive_loss, metrics = self._siglip(graph_normed, text_normed, attack_labels)

        # Auxiliary losses
        aux_loss = torch.tensor(0.0, device=graph_normed.device)
        if compute_auxiliary and self.use_auxiliary_tasks and hasattr(batch, 'y_attack'):
            aux_loss, aux_metrics = self.auxiliary.compute_loss(graph_embs, text_embs, batch)
            metrics.update(aux_metrics)

        total_loss = self.contrastive_weight * contrastive_loss + self.auxiliary_weight * aux_loss
        metrics['total_loss'] = total_loss.item()

        return graph_normed, text_normed, total_loss, metrics

    def _siglip(self, graph_embs, text_embs, attack_labels=None):
        """SigLIP sigmoid contrastive loss (Zhai et al., 2023) with soft targets."""
        B = graph_embs.size(0)
        temp = self.temperature.clamp(min=0.01, max=1.0)
        logits = (graph_embs @ text_embs.T) / temp

        # Build target matrix
        if attack_labels is not None and self.soft_target_alpha > 0:
            # ConGraT-style: same-class off-diagonal gets softer penalty
            class_sim = (attack_labels.unsqueeze(0) == attack_labels.unsqueeze(1)).float()
            identity = torch.eye(B, device=logits.device)
            soft_targets = (1.0 - self.soft_target_alpha) * identity + self.soft_target_alpha * class_sim
            labels = 2.0 * soft_targets - 1.0
        else:
            labels = 2.0 * torch.eye(B, device=logits.device) - 1.0

        # Sigmoid binary cross-entropy per pair
        loss = -F.logsigmoid(labels * logits).mean()

        # Monitoring metrics
        with torch.no_grad():
            hard_targets = torch.arange(B, device=logits.device)
            acc_g2t = (logits.argmax(1) == hard_targets).float().mean().item()
            acc_t2g = (logits.T.argmax(1) == hard_targets).float().mean().item()
            pos_sim = torch.diagonal(logits).mean().item()
            mask = ~torch.eye(B, dtype=bool, device=logits.device)
            neg_sim = logits[mask].mean().item()

        return loss, {
            'contrastive_loss': loss.item(),
            'acc_g2t': acc_g2t, 'acc_t2g': acc_t2g,
            'acc_avg': (acc_g2t + acc_t2g) / 2,
            'pos_sim': pos_sim, 'neg_sim': neg_sim,
            'temperature': temp.item(),
        }

    def save_checkpoint(self, path, epoch, optimizer_state=None):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.state_dict(),
            'config': {
                'hidden_dim': self.hidden_dim, 'pooling': self.pooling,
                'gnn_dim': self.gnn_dim, 'text_dim': self.text_dim,
                'use_auxiliary_tasks': self.use_auxiliary_tasks,
                'contrastive_weight': self.contrastive_weight,
                'auxiliary_weight': self.auxiliary_weight,
                'use_qformer': self.use_qformer,
                'soft_target_alpha': self.soft_target_alpha,
                'version': 'v2.2',
            },
            'optimizer_state_dict': optimizer_state,
        }, path)

# Create model
model = CrossAttentionBridgeV2(
    gnn_model=gnn_model,
    text_encoder=text_encoder,
    hidden_dim=256,
    dropout=0.1,
    pooling='mean',
    use_auxiliary_tasks=True,
    num_attack_classes=num_attack_classes,
    contrastive_weight=0.5,
    auxiliary_weight=0.5,
    use_qformer=True,
    num_queries=4,
    num_qformer_layers=2,
    soft_target_alpha=0.1,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model v2.2 created:")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen (GNN+BERT):    {total_params - trainable_params:,}")
print(f"  QFormer: {sum(p.numel() for p in model.qformer.parameters()):,} params")
print(f"  Loss: SigLIP + soft targets (alpha={model.soft_target_alpha})")

# Forward pass test
model.train()
batch_graphs, batch_texts = next(iter(train_loader))
batch_graphs = batch_graphs.to(device)
g_emb, t_emb, loss, metrics = model(batch_graphs, batch_texts)
print(f"\nForward pass test:")
print(f"  Graph embs: {g_emb.shape}, Text embs: {t_emb.shape}")
print(f"  Loss: {loss.item():.4f}, Acc: {metrics['acc_avg']:.3f}")

Model v2.2 created:
  Total parameters:     2,832,267
  Trainable parameters: 2,093,451
  Frozen (GNN+BERT):    738,816
  QFormer: 1,613,568 params
  Loss: SigLIP + soft targets (alpha=0.1)

Forward pass test:
  Graph embs: torch.Size([32, 256]), Text embs: torch.Size([32, 256])
  Loss: 0.8698, Acc: 0.031


## Section 6: Training
AdamW with warmup + cosine decay, gradient accumulation, per-epoch metrics.

In [26]:
# === Cell 13: Training & Validation Functions ===

def train_epoch(model, loader, optimizer, device, epoch, grad_accum_steps=1):
    model.train()
    total_loss = 0.0
    total_metrics = {}
    num_batches = 0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]")
    for step, (batch_graphs, batch_texts) in enumerate(pbar):
        batch_graphs = batch_graphs.to(device)

        g_emb, t_emb, loss, metrics = model(batch_graphs, batch_texts, compute_auxiliary=True)

        scaled_loss = loss / grad_accum_steps
        scaled_loss.backward()

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item()
        for k, v in metrics.items():
            total_metrics[k] = total_metrics.get(k, 0.0) + v
        num_batches += 1

        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{metrics.get('acc_avg', 0):.3f}"})

    avg = {k: v / num_batches for k, v in total_metrics.items()}
    avg['loss'] = total_loss / num_batches
    return avg


@torch.no_grad()
def validate(model, loader, device, epoch):
    model.eval()
    total_loss = 0.0
    total_metrics = {}
    num_batches = 0
    all_g_embs, all_t_embs, all_labels = [], [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Val]  ")
    for batch_graphs, batch_texts in pbar:
        batch_graphs = batch_graphs.to(device)
        g_emb, t_emb, loss, metrics = model(batch_graphs, batch_texts, compute_auxiliary=True)

        all_g_embs.append(g_emb.cpu())
        all_t_embs.append(t_emb.cpu())
        if hasattr(batch_graphs, 'y_attack'):
            all_labels.append(batch_graphs.y_attack.cpu())

        total_loss += loss.item()
        for k, v in metrics.items():
            total_metrics[k] = total_metrics.get(k, 0.0) + v
        num_batches += 1

        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{metrics.get('acc_avg', 0):.3f}"})

    avg = {k: v / num_batches for k, v in total_metrics.items()}
    avg['loss'] = total_loss / num_batches

    # Full-set retrieval metrics
    all_g = torch.cat(all_g_embs)
    all_t = torch.cat(all_t_embs)
    avg.update(compute_recall_at_k(all_g, all_t))

    if all_labels:
        all_l = torch.cat(all_labels)
        avg.update(compute_zero_shot_accuracy(all_g, all_t, all_l, all_l))
        avg.update(compute_uniformity_alignment(all_g, all_t))

    return avg

print("Training and validation functions defined.")

Training and validation functions defined.


In [37]:
# === Cell 14: Evaluation Metrics ===

def compute_recall_at_k(graph_embs, text_embs, k_values=[1, 5, 10]):
    """Recall@K: is the matching text in the top-K most similar?"""
    N = graph_embs.size(0)
    sim = graph_embs @ text_embs.T
    gt = torch.arange(N, device=graph_embs.device)

    metrics = {}
    for k in k_values:
        k_actual = min(k, N)
        _, topk_g2t = sim.topk(k_actual, dim=1)
        hits_g2t = (topk_g2t == gt.unsqueeze(1)).any(dim=1).float().mean().item()

        _, topk_t2g = sim.T.topk(k_actual, dim=1)
        hits_t2g = (topk_t2g == gt.unsqueeze(1)).any(dim=1).float().mean().item()

        metrics[f"R@{k}_g2t"] = hits_g2t
        metrics[f"R@{k}_t2g"] = hits_t2g
        metrics[f"R@{k}_avg"] = (hits_g2t + hits_t2g) / 2
    return metrics


def compute_zero_shot_accuracy(graph_embs, text_embs, graph_labels, text_labels):
    """Zero-shot: classify graphs by nearest text prototype per class."""
    unique_labels = torch.unique(text_labels)
    prototypes, proto_labels = [], []

    for label in unique_labels:
        mask = text_labels == label
        if mask.sum() > 0:
            proto = F.normalize(text_embs[mask].mean(dim=0), dim=0)
            prototypes.append(proto)
            proto_labels.append(label.item())

    if not prototypes:
        return {"zero_shot_acc": 0.0}

    prototypes_t = torch.stack(prototypes)
    proto_labels_t = torch.tensor(proto_labels, device=graph_embs.device)

    sim = graph_embs @ prototypes_t.T
    predicted = proto_labels_t[sim.argmax(dim=1)]
    correct = (predicted == graph_labels).float()

    metrics = {"zero_shot_acc": correct.mean().item()}
    for label in unique_labels:
        mask = graph_labels == label
        if mask.sum() > 0:
            metrics[f"zero_shot_acc_class_{label.item()}"] = correct[mask].mean().item()
    return metrics


def compute_uniformity_alignment(graph_embs, text_embs, t=2.0):
    """Wang & Isola 2020: alignment (lower=better) + uniformity (lower=better)."""
    alignment = (graph_embs - text_embs).norm(dim=1).pow(2).mean().item()

    N = graph_embs.size(0)
    if N > 1:
        g_pdist = torch.pdist(graph_embs, p=2).pow(2)
        u_graph = torch.log(torch.exp(-t * g_pdist).mean() + 1e-10).item()
        t_pdist = torch.pdist(text_embs, p=2).pow(2)
        u_text = torch.log(torch.exp(-t * t_pdist).mean() + 1e-10).item()
    else:
        u_graph = u_text = 0.0

    return {
        "alignment": alignment,
        "uniformity_graph": u_graph,
        "uniformity_text": u_text,
        "uniformity_avg": (u_graph + u_text) / 2,
    }

print("Metrics defined: compute_recall_at_k, compute_zero_shot_accuracy, compute_uniformity_alignment")

Metrics defined: compute_recall_at_k, compute_zero_shot_accuracy, compute_uniformity_alignment


In [ ]:
# === Cell 15: Training Loop ===

NUM_EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 5
GRAD_ACCUM = 2  # effective batch = 32 * 2 = 64 (SigLIP doesn't need large effective batch)
CHECKPOINT_DIR = 'ANIDC/checkpoints/stage2_77dim'

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY
)

def get_lr_multiplier(epoch, warmup, total):
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / (total - warmup)
    return 0.5 * (1 + math.cos(math.pi * progress))

best_val_loss = float('inf')
best_val_r1 = 0.0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    # LR schedule
    lr_mult = get_lr_multiplier(epoch - 1, WARMUP_EPOCHS, NUM_EPOCHS)
    current_lr = LR * lr_mult
    for pg in optimizer.param_groups:
        pg['lr'] = current_lr

    # Train
    train_m = train_epoch(model, train_loader, optimizer, device, epoch, GRAD_ACCUM)

    # Validate
    val_m = validate(model, val_loader, device, epoch)

    history.append({'epoch': epoch, 'train': train_m, 'val': val_m})

    # Print
    print(f"\nEpoch {epoch}/{NUM_EPOCHS} (lr={current_lr:.6f})")
    print(f"  Train: loss={train_m['loss']:.4f}  acc={train_m.get('acc_avg', 0):.3f}")
    print(f"  Val:   loss={val_m['loss']:.4f}  acc={val_m.get('acc_avg', 0):.3f}")
    if 'R@1_avg' in val_m:
        print(f"  R@1={val_m['R@1_avg']:.3f}  R@5={val_m.get('R@5_avg', 0):.3f}")

    # Checkpoint on best val loss
    if val_m['loss'] < best_val_loss:
        best_val_loss = val_m['loss']
        model.save_checkpoint(f'{CHECKPOINT_DIR}/best_loss.pt', epoch, optimizer.state_dict())
        print(f"  -> Saved best loss checkpoint (loss={best_val_loss:.4f})")

    # Checkpoint on best R@1
    val_r1 = val_m.get('R@1_avg', 0)
    if val_r1 > best_val_r1:
        best_val_r1 = val_r1
        model.save_checkpoint(f'{CHECKPOINT_DIR}/best_r1.pt', epoch, optimizer.state_dict())
        print(f"  -> Saved best R@1 checkpoint (R@1={best_val_r1:.3f})")

# Final save
model.save_checkpoint(f'{CHECKPOINT_DIR}/final.pt', NUM_EPOCHS, optimizer.state_dict())
print(f"\nTraining complete! Best val loss: {best_val_loss:.4f}, Best R@1: {best_val_r1:.3f}")

Epoch 1 [Val]  : 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 75/75 [01:00<00:00,  1.24it/s, loss=0.5646, acc=0.060]



Epoch 1/50 (lr=0.000020)
  Train: loss=0.5748  acc=0.045
  Val:   loss=0.5642  acc=0.042
  R@1=0.001  R@5=0.003
  -> Saved best loss checkpoint (loss=0.5642)
  -> Saved best R@1 checkpoint (R@1=0.001)


Epoch 2 [Val]  : 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 75/75 [00:19<00:00,  3.94it/s, loss=0.5642, acc=0.060]



Epoch 2/50 (lr=0.000040)
  Train: loss=0.4977  acc=0.100
  Val:   loss=0.5557  acc=0.060
  R@1=0.001  R@5=0.007
  -> Saved best loss checkpoint (loss=0.5557)
  -> Saved best R@1 checkpoint (R@1=0.001)


Epoch 3 [Val]  : 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 75/75 [00:15<00:00,  4.82it/s, loss=0.5433, acc=0.040]



Epoch 3/50 (lr=0.000060)
  Train: loss=0.4654  acc=0.140
  Val:   loss=0.5335  acc=0.069
  R@1=0.002  R@5=0.009
  -> Saved best loss checkpoint (loss=0.5335)
  -> Saved best R@1 checkpoint (R@1=0.002)


Epoch 4 [Train]:  40%|█████████████████████████████████████▉                                                          | 138/349 [05:08<08:45,  2.49s/it, loss=0.4213, acc=0.188]

## Section 7: Visualization & Retrieval Demo
t-SNE projection of learned embeddings + interactive retrieval examples.

In [ ]:
# === Cell 16: t-SNE Visualization ===
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Collect embeddings from test set
model.eval()
all_g_embs, all_t_embs, all_labels, all_attack_names = [], [], [], []

with torch.no_grad():
    for batch_graphs, batch_texts in tqdm(test_loader, desc="Extracting test embeddings"):
        batch_graphs = batch_graphs.to(device)
        g_emb, t_emb, _, _ = model(batch_graphs, batch_texts, compute_auxiliary=False)
        all_g_embs.append(g_emb.cpu())
        all_t_embs.append(t_emb.cpu())
        if hasattr(batch_graphs, 'y_attack'):
            all_labels.append(batch_graphs.y_attack.cpu())
        for i in range(len(batch_texts)):
            all_attack_names.append(batch_graphs[i].attack_type if hasattr(batch_graphs[i], 'attack_type') else 'Unknown')

g_embs = torch.cat(all_g_embs).numpy()
t_embs = torch.cat(all_t_embs).numpy()
labels = torch.cat(all_labels).numpy() if all_labels else np.zeros(len(g_embs))

N = len(g_embs)
print(f"Test set: {N} graph-text pairs")

# t-SNE on combined graph + text embeddings
combined_embs = np.concatenate([g_embs, t_embs], axis=0)
perplexity = min(30, len(combined_embs) - 1)
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
tsne_2d = tsne.fit_transform(combined_embs)

g_tsne = tsne_2d[:N]
t_tsne = tsne_2d[N:]

# Get unique labels and color map
unique_labels = sorted(set(labels))
# Map label IDs to attack names
label_id_to_name = test_dataset.attack_type_to_id
name_to_id = label_id_to_name
id_to_name = {v: k for k, v in name_to_id.items()}

colors = plt.cm.tab10(np.linspace(0, 1, max(len(unique_labels), 2)))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Plot 1: All embeddings colored by attack type
ax = axes[0]
for i, label in enumerate(unique_labels):
    mask = labels == label
    name = id_to_name.get(int(label), f'Class {int(label)}')
    ax.scatter(g_tsne[mask, 0], g_tsne[mask, 1], c=[colors[i]], marker='o',
               alpha=0.6, s=30, label=f'{name} (graph)')
    ax.scatter(t_tsne[mask, 0], t_tsne[mask, 1], c=[colors[i]], marker='^',
               alpha=0.6, s=30, label=f'{name} (text)')
ax.set_title('t-SNE: Graph (o) & Text (^) Embeddings by Attack Type', fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8, ncol=1)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')

# Plot 2: Lines connecting matching pairs (subsample for clarity)
ax = axes[1]
max_pairs = min(200, N)
sample_idx = np.random.choice(N, max_pairs, replace=False) if N > max_pairs else np.arange(N)

for i, label in enumerate(unique_labels):
    mask = labels == label
    name = id_to_name.get(int(label), f'Class {int(label)}')
    ax.scatter(g_tsne[mask, 0], g_tsne[mask, 1], c=[colors[i]], marker='o', alpha=0.5, s=20)
    ax.scatter(t_tsne[mask, 0], t_tsne[mask, 1], c=[colors[i]], marker='^', alpha=0.5, s=20, label=name)

# Draw lines between matching graph-text pairs
for idx in sample_idx:
    ax.plot([g_tsne[idx, 0], t_tsne[idx, 0]],
            [g_tsne[idx, 1], t_tsne[idx, 1]],
            c='gray', alpha=0.15, linewidth=0.5)

ax.set_title(f't-SNE: Matching Pairs Connected ({max_pairs} shown)', fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/tsne_stage2.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {CHECKPOINT_DIR}/tsne_stage2.png")

In [29]:
# === Cell 16.5: Load Model from Checkpoint ===
# Use this cell to load a trained model without re-running training.

CHECKPOINT_DIR = 'ANIDC/checkpoints/stage2_77dim'

# Pick best checkpoint (prefer best_recall > best_loss > final)
for ckpt_name in ['best.pt', 'best_recall.pt', 'best_loss.pt', 'final.pt']:
    ckpt_path = os.path.join(CHECKPOINT_DIR, ckpt_name)
    if os.path.exists(ckpt_path):
        break
else:
    raise FileNotFoundError(f"No checkpoint found in {CHECKPOINT_DIR}/")

print(f"Loading checkpoint: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f"  Epoch: {ckpt['epoch']}")
print(f"  Config: {ckpt['config']}")

# Reconstruct model with saved config
cfg = ckpt['config']
model = CrossAttentionBridgeV2(
    gnn_model=gnn_model,
    text_encoder=text_encoder,
    hidden_dim=cfg.get('hidden_dim', 256),
    dropout=0.1,
    pooling=cfg.get('pooling', 'mean'),
    use_auxiliary_tasks=cfg.get('use_auxiliary_tasks', True),
    num_attack_classes=num_attack_classes,
    contrastive_weight=cfg.get('contrastive_weight', 0.5),
    auxiliary_weight=cfg.get('auxiliary_weight', 0.5),
    use_qformer=cfg.get('use_qformer', False),
    num_queries=4,
    num_qformer_layers=2,
    soft_target_alpha=cfg.get('soft_target_alpha', 0.1),
).to(device)

model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"\nModel loaded and set to eval mode.")
print(f"  QFormer: {cfg.get('use_qformer', False)}")
print(f"  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Loading checkpoint: ANIDC/checkpoints/stage2_77dim/best.pt
  Epoch: 9
  Config: {'hidden_dim': 256, 'pooling': 'mean', 'gnn_dim': 128, 'text_dim': 768, 'use_auxiliary_tasks': True, 'contrastive_weight': 0.5, 'auxiliary_weight': 0.5, 'version': 'v2.1'}

Model loaded and set to eval mode.
  QFormer: False
  Trainable params: 447,115


In [30]:
# === Cell 17: Retrieval Demo ===

def retrieve_top_k(model, query_type, query, candidates, candidate_loader, k=5):
    """Retrieve top-k matches.

    query_type: 'text' (query is a string, retrieve graphs) or 'graph' (query is a Batch, retrieve texts)
    """
    model.eval()
    with torch.no_grad():
        if query_type == 'text':
            q_emb = model.text_encoder.encode([query])
            q_emb = model.text_proj(q_emb)
            q_emb = F.normalize(q_emb, dim=-1)  # [1, 256]

            # Encode all candidate graphs
            all_embs, all_info = [], []
            for batch_graphs, batch_texts in candidate_loader:
                batch_graphs = batch_graphs.to(device)
                node_embs = model.gnn.encode(batch_graphs.x, batch_graphs.edge_index)
                # Use QFormer if available, otherwise mean pool
                if hasattr(model, 'use_qformer') and model.use_qformer:
                    graph_embs = model.qformer(node_embs, batch_graphs.batch)
                else:
                    graph_embs = model.pool_graph(node_embs, batch_graphs.batch)
                graph_embs = model.graph_proj(graph_embs)
                graph_embs = F.normalize(graph_embs, dim=-1)
                all_embs.append(graph_embs.cpu())
                for i, txt in enumerate(batch_texts):
                    at = batch_graphs[i].attack_type if hasattr(batch_graphs[i], 'attack_type') else '?'
                    all_info.append({'text': txt, 'attack_type': at,
                                     'nodes': batch_graphs[i].num_nodes,
                                     'edges': batch_graphs[i].edge_index.size(1)})

            all_embs = torch.cat(all_embs)
            sims = (q_emb.cpu() @ all_embs.T).squeeze(0)
            topk_scores, topk_idx = sims.topk(k)
            results = [(all_info[i], topk_scores[j].item()) for j, i in enumerate(topk_idx.tolist())]
            return results

        elif query_type == 'graph':
            query = query.to(device)
            node_embs = model.gnn.encode(query.x, query.edge_index)
            # Use QFormer if available, otherwise mean pool
            if hasattr(model, 'use_qformer') and model.use_qformer:
                graph_embs = model.qformer(node_embs, query.batch)
            else:
                graph_embs = model.pool_graph(node_embs, query.batch)
            q_emb = model.graph_proj(graph_embs)
            q_emb = F.normalize(q_emb, dim=-1)  # [1, 256]

            all_embs, all_texts = [], []
            for batch_graphs, batch_texts in candidate_loader:
                batch_graphs = batch_graphs.to(device)
                t_emb = model.text_encoder.encode(batch_texts)
                t_emb = model.text_proj(t_emb)
                t_emb = F.normalize(t_emb, dim=-1)
                all_embs.append(t_emb.cpu())
                all_texts.extend(batch_texts)

            all_embs = torch.cat(all_embs)
            sims = (q_emb.cpu() @ all_embs.T).squeeze(0)
            topk_scores, topk_idx = sims.topk(k)
            results = [(all_texts[i], topk_scores[j].item()) for j, i in enumerate(topk_idx.tolist())]
            return results


# === Demo 1: Text → Graph retrieval ===
print("=" * 80)
print("TEXT → GRAPH RETRIEVAL")
print("=" * 80)

queries = [
    "Denial of Service attack with high-degree flooding behavior",
    "Normal benign traffic with standard communication patterns",
    "Port scanning activity probing multiple network hosts",
]

for query_text in queries:
    print(f"\nQuery: \"{query_text}\"")
    results = retrieve_top_k(model, 'text', query_text, None, test_loader, k=5)
    for rank, (info, score) in enumerate(results, 1):
        print(f"  #{rank} (sim={score:.3f}) [{info['attack_type']}] "
              f"{info['nodes']} nodes, {info['edges']} edges")

# === Demo 2: Graph → Text retrieval ===
print("\n" + "=" * 80)
print("GRAPH → TEXT RETRIEVAL")
print("=" * 80)

# Pick 3 graphs from different classes
demo_graphs = []
seen_classes = set()
for batch_graphs, batch_texts in test_loader:
    for i in range(batch_graphs.num_graphs):
        g = batch_graphs[i]
        cls = g.y_attack.item() if hasattr(g, 'y_attack') else -1
        if cls not in seen_classes and len(demo_graphs) < 3:
            seen_classes.add(cls)
            demo_graphs.append((g, batch_texts[i], cls))
    if len(demo_graphs) >= 3:
        break

for g, original_text, cls in demo_graphs:
    query_batch = Batch.from_data_list([g])
    at_name = id_to_name.get(cls, f'Class {cls}')
    print(f"\nQuery graph: [{at_name}] {g.num_nodes} nodes, {g.edge_index.size(1)} edges")
    print(f"  Original text: {original_text[:80]}...")

    results = retrieve_top_k(model, 'graph', query_batch, None, test_loader, k=5)
    for rank, (text, score) in enumerate(results, 1):
        print(f"  #{rank} (sim={score:.3f}) {text[:80]}...")

TEXT → GRAPH RETRIEVAL

Query: "Denial of Service attack with high-degree flooding behavior"
  #1 (sim=0.306) [WebAttack] 149 nodes, 225 edges
  #2 (sim=0.276) [WebAttack] 100 nodes, 188 edges
  #3 (sim=0.275) [WebAttack] 242 nodes, 335 edges
  #4 (sim=0.272) [Benign] 21 nodes, 37 edges
  #5 (sim=0.268) [Botnet] 6 nodes, 8 edges

Query: "Normal benign traffic with standard communication patterns"
  #1 (sim=0.161) [Benign] 34 nodes, 53 edges
  #2 (sim=0.159) [Benign] 5 nodes, 9 edges
  #3 (sim=0.158) [Benign] 25 nodes, 68 edges
  #4 (sim=0.158) [Benign] 36 nodes, 93 edges
  #5 (sim=0.156) [Benign] 4 nodes, 6 edges

Query: "Port scanning activity probing multiple network hosts"
  #1 (sim=0.181) [Benign] 12 nodes, 22 edges
  #2 (sim=0.179) [Botnet] 20 nodes, 37 edges
  #3 (sim=0.172) [WebAttack] 20 nodes, 34 edges
  #4 (sim=0.162) [WebAttack] 66 nodes, 101 edges
  #5 (sim=0.161) [Benign] 7 nodes, 10 edges

GRAPH → TEXT RETRIEVAL


NameError: name 'id_to_name' is not defined

## Section 8: Save Final Checkpoint
Save best model with full config for downstream use.

In [9]:
# === Cell 18: Save Final Model ===

# Load best model (by R@1) and re-save as canonical best.pt
best_ckpt_path = f'{CHECKPOINT_DIR}/best_recall.pt'
if os.path.exists(best_ckpt_path):
    best_src = best_ckpt_path
    print(f"Using best R@1 checkpoint: {best_src}")
else:
    best_src = f'{CHECKPOINT_DIR}/best_loss.pt'
    print(f"Using best loss checkpoint: {best_src}")

# Copy best as canonical best.pt
import shutil
canonical_path = f'{CHECKPOINT_DIR}/best.pt'
shutil.copy2(best_src, canonical_path)

# Print summary
ckpt = torch.load(canonical_path, map_location='cpu', weights_only=False)
print(f"\nSaved canonical checkpoint: {canonical_path}")
print(f"  Epoch: {ckpt['epoch']}")
print(f"  Config: {ckpt['config']}")

# List all saved checkpoints
print(f"\nAll checkpoints in {CHECKPOINT_DIR}/:")
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    if f.endswith('.pt') or f.endswith('.png'):
        fpath = os.path.join(CHECKPOINT_DIR, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {f}: {size_mb:.1f} MB")

print(f"\nStage 2 pipeline complete!")
print(f"  Model: CLIP-style contrastive bridge (v2.1)")
print(f"  Best R@1: {best_val_r1:.3f}")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"\nNext steps:")
print(f"  1. Augment train.jsonl with Flan-T5 (02_augment_with_flan_kaggle.py)")
print(f"  2. Re-run training with augmented data")
print(f"  3. Evaluate on zero-shot attack detection")

Using best R@1 checkpoint: ANIDC/checkpoints/stage2_77dim/best_recall.pt

Saved canonical checkpoint: ANIDC/checkpoints/stage2_77dim/best.pt
  Epoch: 9
  Config: {'hidden_dim': 256, 'pooling': 'mean', 'gnn_dim': 128, 'text_dim': 768, 'use_auxiliary_tasks': True, 'contrastive_weight': 0.5, 'auxiliary_weight': 0.5, 'version': 'v2.1'}

All checkpoints in ANIDC/checkpoints/stage2_77dim/:
  best.pt: 8.0 MB
  best_loss.pt: 26.9 MB
  best_r1.pt: 26.9 MB
  best_recall.pt: 8.0 MB
